# **DDDM - Mini - Project 1 - Data Wrangling: Ladywood in Context**

Ladywood is a ward in inner-city Birmingham, and the subject of this year's
Engineering for People Design Challenge. Before anyone designs anything for a
place, they need to understand it, and the first question is always the same:
**how is this place different from the places around it?**

This project answers that using the 2021 Census. You will work with real data
published by the Office for National Statistics, at the smallest geography it is
released for, and by the end you will have reproduced several of the statistics
quoted in the design brief itself.

The data covers all 35,672 Lower Layer Super Output Areas in England and Wales.
An LSOA holds roughly 1,500 people. Birmingham is one local authority within
that, and Ladywood is one ward within Birmingham. Working out how many LSOAs
Birmingham has is the first thing the project asks you to do.

**This project has hidden tests.** Passing every visible test does not guarantee
full marks.

**If a test fails**, check three things before anything else: that you spelled
the variable name exactly as the question asked, that you ran the cell, and that
you have run every cell above it from the top. Most failures are one of those.

> **This is version 2 of the project.** If you still have the earlier copy open, use this one instead: the first cell below installs the libraries, which the earlier copy was missing. Nothing else has changed, so any work you have done can be pasted across.


## **Do not change the code in the two cells below** ##

In [ ]:
# Run this cell first to install the libraries. It may take a minute.
%pip install -q sphinx<9.2
%pip install -q urllib3<2.0 otter-grader==7.0.0 datascience ipywidgets

In [ ]:
# Run this cell to set up the notebook, but please don't change it.

try:
    import pyodide_http
    pyodide_http.patch_all()
except ImportError:
    pass

import numpy as np
from datascience import *

import matplotlib
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
warnings.simplefilter('ignore', FutureWarning)

import otter
grader = otter.Notebook("project1-v2.ipynb")

# The helper for the 'Just run this cell' cells lives in project_helpers.py,
# which must sit in the same folder as this notebook.
try:
    from project_helpers import needs, checkpoint, summary
except ImportError:
    raise ImportError(
        'project_helpers.py was not found. It must be in the same folder as this '
        'notebook. Download it from Ulwazi alongside the notebook and the CSV '
        'files, then run this cell again.')


---

## **1. Getting the data in**

Real data does not arrive ready to use. The census files have more rows than you
want, more columns than you want, and column names nobody would choose.

Start by looking at what you have.

In [ ]:
# Just run this cell
tenure_raw = Table.read_table('census2021-ts054-lsoa.csv')
print(tenure_raw.num_rows, 'rows,', tenure_raw.num_columns, 'columns')
tenure_raw.show(3)

Every row is one LSOA in England or Wales. The `geography` column names it, and
`geography code` is its official identifier, which looks like `E01012101`.

---

**Question 1.1.** The full table covers England and Wales. Assign `birmingham_tenure` to a table containing only the rows whose `geography` name contains `'Birmingham'`, then assign `n_birmingham` to the number of rows in it. **(8 Points)**

*The `where` call is written for you; you supply what goes inside it.* Its first argument is a column name, and `are.containing` takes the text to look for. `n_birmingham` has no call written: it is a count of the rows in the table you have just built.

In [ ]:
birmingham_tenure = tenure_raw.where(..., are.containing(...))
n_birmingham = ...
n_birmingham

In [ ]:
grader.check("q1_1")

In [ ]:
# Just run this cell
checkpoint('q1_1')


---

Those column names are unusable, so the next cell cuts the table down to four
columns and renames them.

| Original | New name |
|---|---|
| `geography code` | `LSOA` |
| `Tenure of household: Total: All households` | `Households` |
| `Tenure of household: Owned` | `Owned` |
| `Tenure of household: Private rented` | `Private rented` |

Read it before you run it. At Question 5.1 the `select` is written out for you again, but the renaming is not: you write that part yourself.
`select` keeps the columns you name and discards the rest; each `relabeled` call
takes the old name and the new one.

In [ ]:
# Just run this cell
with needs('Question 1.1'):
    tenure = (birmingham_tenure
              .select('geography code',
                      'Tenure of household: Total: All households',
                      'Tenure of household: Owned',
                      'Tenure of household: Private rented')
              .relabeled('geography code', 'LSOA')
              .relabeled('Tenure of household: Total: All households', 'Households')
              .relabeled('Tenure of household: Owned', 'Owned')
              .relabeled('Tenure of household: Private rented', 'Private rented'))
    tenure.show(3)

---

## **2. Which ward is which?**

The census tells you about LSOAs, but nobody thinks in LSOAs. Ladywood is a
**ward**, and a ward is made of several LSOAs.

The lookup table below maps each LSOA to its ward and local authority. This is
the join that turns a table of anonymous codes into something you can reason
about.

---

**Question 2.1.** Load `lookup.csv` into a table called `lookup_raw`. Then assign `lookup` to a table with just three columns, renamed as below. **(8 Points)**

| Original | New name |
|---|---|
| `LSOA21CD` | `LSOA` |
| `WD25NM` | `Ward` |
| `LAD25NM` | `Council` |

*Again the shape is given and the arguments are not.* Each `relabeled` call takes the old name first, then the new one. The cell above shows the same pattern with everything filled in.

In [ ]:
lookup_raw = Table.read_table(...)
lookup = (lookup_raw
          .select(..., ..., ...)
          .relabeled(..., ...)
          .relabeled(..., ...)
          .relabeled(..., ...))
lookup.show(3)

In [ ]:
grader.check("q2_1")

In [ ]:
# Just run this cell
checkpoint('q2_1')


---

### **A worked example: `join`**

`join` lines up two tables on a shared column. The four made-up rows below show
exactly what it does, and what it quietly does **not** do, before you use it on
the real tables.


In [ ]:
# Just run this cell
counts = Table().with_columns(
    'code',       make_array('E01', 'E02', 'E03', 'E04'),
    'households', make_array(100, 200, 300, 400))

names = Table().with_columns(
    'code', make_array('E01', 'E02', 'E03'),
    'town', make_array('Ladywood', 'Aston', 'Nechells'))

matched = counts.join('code', names, 'code')

print('counts  has', counts.num_rows, 'rows')
print('names   has', names.num_rows, 'rows')
print('matched has', matched.num_rows, 'rows')
print('E04 was in counts but not in names, so the join dropped it.')
print('Rows lost:', counts.num_rows - matched.num_rows)
print()
print('Distinct towns, using group:', matched.group('town').num_rows)
matched.show()


Three things to take from that cell.

1. `counts.join('code', names, 'code')` reads as: my column, the other table,
   their column. Where both tables use the same name, the third argument can be
   left out: `counts.join('code', names)`.
2. **The join lost a row and did not warn you.** `E04` had no match, so it is
   simply gone. Subtracting `num_rows` after from `num_rows` before is the only
   way you find out.
3. `group` gives one row per distinct value, so `group(label).num_rows` counts
   the distinct values in a column.

---


**Question 2.2.** Join `tenure` to `lookup` on the `LSOA` column and assign the result to `tenure_wards`. **(12 Points)**

Then assign:

1. `rows_lost` to the number of rows `tenure` had that did not survive the join
2. `smallest_ward_size` to the number of LSOAs in the **smallest** ward, meaning the fewest any single ward has

*The worked example above does the join and the lost rows on four made-up rows.* Here you do them on 659. The shape of the join is given and the arguments are not.

`smallest_ward_size` is the new one. The worked example counted how many distinct towns there were; this asks how large the smallest group is, which is a different question about the same grouped table. Look at what `group` actually gives you back.

*A real warning, not a hint:* a join that drops rows gives you a wrong answer that looks perfectly reasonable. `rows_lost` is in this question because checking it should become a habit, not because the marks need padding.

*Why the smallest ward matters:* every ward average you compute later rests on however many LSOAs that ward contains, and they do not all contain the same number. A ward built from few of them gives a shakier average than one built from many.

In [ ]:
tenure_wards       = tenure.join(..., ...)
rows_lost          = ...
smallest_ward_size = ...

print('Rows lost to the join:', rows_lost)
print('Distinct wards:', tenure_wards.group('Ward').num_rows)
print('Smallest ward:', smallest_ward_size, 'LSOAs')

In [ ]:
grader.check("q2_2")

---

## **3. Ladywood against the city**

Now the question the design brief is really asking.

---

**Question 3.1.** The first line of the cell below gives you Ladywood's rows. Work out how much more common private renting is in Ladywood than in Birmingham as a whole. **(12 Points)**

Assign:

1. `ladywood_private_pct` to the percentage of Ladywood households that rent privately
2. `birmingham_private_pct` to the same figure for the whole of Birmingham
3. `private_rented_ratio` to the first divided by the second

*Note:* You cannot average the LSOA percentages, because the LSOAs have different numbers of households. Add the counts first, then divide once. This applies to both percentages, not just the first.

*The Ladywood lines are laid out for you; the Birmingham ones are not.* They are the same three steps on the whole table, so work from the pattern above them.

*Hint:* `tenure_wards` already contains only Birmingham, since **Question 1.1** filtered it.

In [ ]:
ladywood = tenure_wards.where('Ward', 'Ladywood')
print('Ladywood LSOAs:', ladywood.num_rows)

ladywood_renters       = sum(ladywood.column(...))
ladywood_households    = sum(ladywood.column(...))
ladywood_private_pct   = ... / ... * 100

birmingham_renters     = sum(tenure_wards.column(...))     
birmingham_households  = sum(tenure_wards.column(...))
birmingham_private_pct = ... / ... * 100

private_rented_ratio   = ... / ...
private_rented_ratio

In [ ]:
grader.check("q3_1")

Private renting is only one of the columns you kept. `Owned` has been sitting in
`tenure` since section 1. Run the cell below to put the two side by side.

In [ ]:
# Just run this cell
summary('q3_1')

Ownership is the sharper contrast of the two. Ladywood has the lowest home
ownership of any ward in Birmingham, and the gap from the city average is wider
than the gap in private renting.

Notice the third number. The census counts five tenures and you selected two of
them, so about a third of Ladywood's households are in neither column, almost
all of them renting from the council or a housing association. "Not an owner"
and "private renter" are different claims about a place, and a design proposal
that confuses them will get the wrong answer.

---

## **4. Every ward at once**

Comparing one ward to the city average is a start. Comparing every ward tells
you whether Ladywood is unusual or typical.

---

### A worked example: grouping with `sum`

You have used `group` with one argument, which counts rows. Given a second
argument it does something different: it applies that function to the other
columns instead of counting. The four made-up rows below show what that looks
like when the table has more than one column left, and the one thing it does to
the column names.

In [ ]:
# Just run this cell
demo = Table().with_columns(
    'ward',       make_array('Ladywood', 'Ladywood', 'Aston', 'Aston'),
    'council',    make_array('Birmingham', 'Birmingham', 'Birmingham', 'Birmingham'),
    'households', make_array(100, 200, 300, 400),
    'rented',     make_array(40, 50, 60, 30))

print('labels before grouping:', demo.labels)
print()
print('one argument, so it counts the rows in each ward:')
demo.group('ward').show()

print('a second argument, so it adds up every other column instead:')
demo.group('ward', sum).show()

print("'council' holds text, so summing it gives a blank. Drop it first:")
demo_totals = demo.drop('council').group('ward', sum)
demo_totals.show()
print('labels after grouping: ', demo_totals.labels)

Three things to take from that cell.

1. `demo.group('ward', sum)` applies `sum` to **every** column except the one
   you grouped by, not just one of them. One row comes back per ward.
2. **Text columns do not add up.** `council` came back blank rather than
   raising an error, so nothing tells you it went wrong except looking. Drop
   the text columns before grouping.
3. `group` **renames** every column it aggregates. `households` became
   `households sum`. Anything you write afterwards has to use the new name.

---

**Question 4.1.** Group `tenure_wards` by `Ward`, summing the numerical columns, and assign the result to `by_ward`. **(8 Points)**

*The calls are written for you; the arguments are the blanks.*

*The worked example above does this on four made-up rows,* with one text column to drop. Here you do it on 659 rows, and there are two text columns, `LSOA` and `Council`.

*Watch out:* the renaming is not cosmetic. Question 4.2 needs the new names, so run `by_ward.labels` once you have an answer and read what came back.

In [ ]:
numbers_only = tenure_wards.drop(..., ...)
by_ward      = numbers_only.group(..., ...)
by_ward.show(3)

In [ ]:
grader.check("q4_1")

---

**Question 4.2.** Add a column called `Private rented %` to `by_ward`, giving each ward's privately renting households as a percentage of that ward's total households, and assign the new table to `by_ward_pct`. **(16 Points)**

Then assign `top_private_ward` to the name of the ward with the highest **`Private rented %`**.

*Watch out:* that is not the same as the ward with the largest **number** of privately renting households. A large ward can have more renters than anywhere else and still have a smaller share of its households renting. Those two questions have different answers here, and only one of them is being asked.

*You did the same arithmetic at Question 3.1*, on one ward rather than all of them. The calls are written out; every blank is a column name or a label.

*Hint:* `tbl.with_columns(name, values)` adds a column, and its first argument is the name as a string.

In [ ]:
rented_pct       = by_ward.column(...) / by_ward.column(...) * 100
by_ward_pct      = by_ward.with_columns(..., rented_pct)
top_private_ward = by_ward_pct.sort(..., descending=True).column(...).item(0)
top_private_ward

In [ ]:
grader.check("q4_2")

Run the cell below to see the distribution across every ward, with Ladywood
marked in red. It draws this, which is the picture from Lab 4 used for a
different question: there the red dot was an observed statistic against a null
distribution, here it is one ward against all the others.

```python
by_ward_pct.hist('Private rented %', bins=np.arange(0, 60, 5), unit='%')
ladywood_pct = by_ward_pct.where('Ward', 'Ladywood').column('Private rented %').item(0)
plots.scatter(ladywood_pct, 0, color='red', s=60, zorder=3)
plots.title('Private renting across Birmingham wards')
plots.show()
```


In [ ]:
# Just run this cell
summary('q4_2')

---

## **5. Cars, or the lack of them**

The design brief says Ladywood residents rely heavily on walking and public
transport. The census does not measure walking or buses, but it does count cars,
and that has direct consequences for anything you might design.

---

**Question 5.1.** Load `census2021-ts045-lsoa.csv`, cut it down to the three columns below, rename them, and join it to `lookup`. Assign the result to `cars`. **(12 Points)**

| Original | New name |
|---|---|
| `geography code` | `LSOA` |
| `Number of cars or vans: Total: All households` | `Households` |
| `Number of cars or vans: No cars or vans in household` | `No car` |

*You built the select-and-rename chain at Question 2.1 and the join at Question 2.2,* both with the shape laid out and the arguments blank. The rename chain is not written out a second time: `cars_named` is one blank, and the three renames inside it are yours. Remember that `select` runs before the renaming, so it needs the original names.

In [ ]:
cars_raw   = Table.read_table(...)
cars_cut   = cars_raw.select(..., ..., ...)
cars_named = ...          # cars_cut with all three columns renamed, as at Question 2.1
cars       = cars_named.join(..., ...)
cars.show(3)

In [ ]:
grader.check("q5_1")

In [ ]:
# Just run this cell
checkpoint('q5_1')


---

**Question 5.2.** Cut `cars` down to Birmingham, group it by ward, add a `No car %` column, and assign the table to `cars_by_ward`. Then assign `ladywood_no_car_pct` to Ladywood's `No car %`. **(12 Points)**

*Note:* `cars` still covers all of England and Wales, so the filter has to come first. Use the `Council` column the lookup gave you.

*This is Questions 4.1 and 4.2 again, on the other dataset,* using the same methods in the same order, with one extra step at the top to cut `cars` down to Birmingham. The blanks do not all sit where they sat there, so read each line. Keep the names the cell gives you, `cars_bham` especially, because the map at the end of the project uses it.

*Hint:* `group` renames what it aggregates, so look at the labels of your grouped table before writing the percentage.

In [ ]:
cars_bham           = cars.where(..., ...)
cars_numbers        = cars_bham.drop(..., ...)
cars_totals         = cars_numbers.group(..., ...)
cars_by_ward        = cars_totals.with_columns('No car %',
                          cars_totals.column(...) / cars_totals.column(...) * 100)
ladywood_no_car_pct = cars_by_ward.where(..., ...).column(...).item(0)
ladywood_no_car_pct

In [ ]:
grader.check("q5_2")

---

**Question 5.3.** Where does Ladywood rank among Birmingham's wards for households without a car? Assign `ladywood_no_car_rank` to its position when the wards are ordered from highest `No car %` to lowest. **(6 Points)**

*This is the first question where every line of the answer is blank.* No call is written and no shape is given: the last two questions are yours to write from scratch.

*Hint:* Counting starts at 1, so the ward with the highest percentage has rank 1, not rank 0.

In [ ]:
ladywood_no_car_rank = ...
ladywood_no_car_rank

In [ ]:
grader.check("q5_3")

---

**Question 5.4.** You have now measured car ownership in Ladywood and compared it with every other ward in Birmingham.

Which of the following is the most defensible conclusion to draw from that alone? Assign `transport_conclusion` to 1, 2 or 3. **(6 Points)**

1. Ladywood residents prefer not to own cars.
2. Car ownership in Ladywood is much lower than in Birmingham as a whole, though this data does not tell us why.
3. Ladywood has better public transport than the rest of Birmingham.

*Think about what the census measured, and what it did not.*

In [ ]:
transport_conclusion = ...
transport_conclusion

In [ ]:
grader.check("q5_4")

---

Numbers in a table are one thing. Run the cell below to see where these
neighbourhoods actually are.

Each dot is one LSOA, placed at its population-weighted centre. Red means more
than 35% of households have no car. Ladywood's are circled.

In [ ]:
# Just run this cell
summary('q5_2')

Car-free neighbourhoods are not scattered at random. They form a band across the
inner city, and Ladywood sits in the middle of it. Whatever is going on here is
about **where** a place is, not just what its households happen to own.

---

## **Findings**

Ladywood is not a smaller version of Birmingham. On every measure in this
project it sits at one end of the city's range. Around half of its households
rent privately, against roughly a fifth citywide. Fewer than one in five own
their home, against more than half citywide. And it has the **highest
proportion of car-free households of any ward in Birmingham**.

That is what makes it a design challenge rather than an engineering exercise. A
proposal that assumes people own cars, or own their homes, will not work here.

The census tells you what is, and says nothing about why. For that you need the
rest of the brief, and the people who live there.

---

## **You're done!**

**Important submission information:**
- **Run all the tests** and verify that they all pass
- **Save** from the **File** menu
- **Run the last cell**, `grader.check_all()`, and check every question is listed
- **Click the download button to download the .ipynb file**
- **Then, go to Ulwazi and submit the .ipynb file to DDDM - Mini - Project 1 - Data Wrangling: Ladywood in Context**

**It is your responsibility to make sure your work is saved before running the last cell.**

<img src="download_ipynb.png" alt="download ipynb" width="800"/>

In [ ]:
grader.check_all()

## **Submission**

Make sure you have run all cells in your notebook in order before clicking the download button to download the .ipynb file. **Please save your work before downloading!**